# Forecasting Exchange Rates using Time Series Analysis (ARIMA vs Exponential Smoothing)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.stats.diagnostic import acorr_ljungbox

from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
df = pd.read_csv('exchange_rate.csv')
df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y %H:%M')
df = df.sort_values('date').reset_index(drop=True)
df.set_index('date', inplace=True)
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Ex_rate'])
plt.title('USD to Australian Dollar Exchange Rate (1990-2010)')
plt.xlabel('Date')
plt.ylabel('Exchange Rate')
plt.show()

In [ ]:
df = df.asfreq('D')
# Forward-fill missing values: on days with no trade/quote (e.g. weekends, public
# holidays), the exchange rate is carried forward from the last known trading day.
# This is standard practice for financial time series, since it reflects the actual
# information available at each point in time (no rate observed = rate unchanged from
# the last quote), unlike linear interpolation, which would implicitly assume smooth,
# continuous movement across gaps that didn't really happen.
df['Ex_rate'] = df['Ex_rate'].ffill()
df.isnull().sum()

In [ ]:
series = df['Ex_rate']

decompose_window = 365
rolling_mean = series.rolling(window=decompose_window).mean()
rolling_std = series.rolling(window=decompose_window).std()

plt.figure(figsize=(14, 5))
plt.plot(series, label='Exchange Rate')
plt.plot(rolling_mean, label='Rolling Mean (365d)', color='red')
plt.plot(rolling_std, label='Rolling Std (365d)', color='green')
plt.legend()
plt.title('Trend and Volatility of Exchange Rate')
plt.show()

In [ ]:
train_size = len(series) - 30
train, test = series.iloc[:train_size], series.iloc[train_size:]
train.shape, test.shape

## Part 2: Model Building - ARIMA

In [ ]:
result_level = adfuller(train)
print('ADF Statistic:', result_level[0])
print('p-value:', result_level[1])

In [ ]:
train_diff = train.diff().dropna()

result_diff = adfuller(train_diff)
print('ADF Statistic (1st diff):', result_diff[0])
print('p-value (1st diff):', result_diff[1])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(train_diff, ax=axes[0], lags=40)
plot_pacf(train_diff, ax=axes[1], lags=40, method='ywm')
axes[0].set_title('ACF of Differenced Series')
axes[1].set_title('PACF of Differenced Series')
plt.show()

In [ ]:
p, d, q = 2, 1, 2
arima_model = ARIMA(train, order=(p, d, q))
arima_fit = arima_model.fit()
print(arima_fit.summary())

### Diagnostics

In [ ]:
residuals = arima_fit.resid

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(residuals)
axes[0].set_title('ARIMA Residuals')
plot_acf(residuals, ax=axes[1], lags=40)
axes[1].set_title('ACF of Residuals')
plt.show()

In [ ]:
lb_test = acorr_ljungbox(residuals, lags=[10, 20], return_df=True)
lb_test

### Forecasting

In [ ]:
arima_forecast = arima_fit.forecast(steps=len(test))
arima_forecast.index = test.index

plt.figure(figsize=(14, 5))
plt.plot(train.iloc[-100:], label='Train (last 100)')
plt.plot(test, label='Actual', color='green')
plt.plot(arima_forecast, label='ARIMA Forecast', color='red')
plt.legend()
plt.title('ARIMA Forecast vs Actual')
plt.show()

## Part 2b: Model Building - Exponential Smoothing

### Parameter Optimization (Grid Search on AIC)

Rather than fixing `trend='add', damped_trend=True` upfront, we grid-search over the combinations of `trend` (additive, multiplicative, or none) and `damped_trend` (True/False) that Holt's method supports, fit each on the training set, and select the combination with the **lowest AIC** (Akaike Information Criterion). AIC balances goodness-of-fit against model complexity, so this avoids arbitrarily picking a configuration and lets the data decide which trend specification best matches the underlying series.

In [ ]:
trend_options = ['add', 'mul', None]
damped_options = [True, False]

es_grid_results = []
best_aic = np.inf
best_params = None
best_es_fit = None

for trend in trend_options:
    # damped_trend is only a valid option when a trend component is present
    damped_choices = damped_options if trend is not None else [False]
    for damped in damped_choices:
        try:
            candidate_model = ExponentialSmoothing(
                train, trend=trend, seasonal=None, damped_trend=damped
            )
            candidate_fit = candidate_model.fit()
            aic = candidate_fit.aic
            es_grid_results.append({'trend': trend, 'damped_trend': damped, 'AIC': aic})

            if aic < best_aic:
                best_aic = aic
                best_params = (trend, damped)
                best_es_fit = candidate_fit
        except Exception as e:
            es_grid_results.append({'trend': trend, 'damped_trend': damped, 'AIC': None})

es_grid_df = pd.DataFrame(es_grid_results).sort_values('AIC', na_position='last').reset_index(drop=True)
print(es_grid_df)
print(f"\nBest configuration: trend={best_params[0]!r}, damped_trend={best_params[1]}  (AIC={best_aic:.2f})")

In [ ]:
# Use the AIC-selected best model for forecasting
es_fit = best_es_fit
es_forecast = es_fit.forecast(steps=len(test))
es_forecast.index = test.index

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(train.iloc[-100:], label='Train (last 100)')
plt.plot(test, label='Actual', color='green')
plt.plot(es_forecast, label='Exponential Smoothing Forecast', color='orange')
plt.legend()
plt.title('Exponential Smoothing Forecast vs Actual')
plt.show()

## Part 3: Evaluation and Comparison

In [ ]:
def mape(actual, predicted):
    actual, predicted = np.array(actual), np.array(predicted)
    return np.mean(np.abs((actual - predicted) / actual)) * 100

def evaluate_forecast(actual, predicted, model_name):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape_val = mape(actual, predicted)
    return {'Model': model_name, 'MAE': mae, 'RMSE': rmse, 'MAPE (%)': mape_val}

In [ ]:
results = []
results.append(evaluate_forecast(test, arima_forecast, 'ARIMA'))
results.append(evaluate_forecast(test, es_forecast, 'Exponential Smoothing'))

results_df = pd.DataFrame(results)
results_df

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(test, label='Actual', color='black', linewidth=2)
plt.plot(arima_forecast, label='ARIMA Forecast', color='red')
plt.plot(es_forecast, label='Exponential Smoothing Forecast', color='orange')
plt.legend()
plt.title('Model Comparison: ARIMA vs Exponential Smoothing')
plt.show()

## Conclusion

Both ARIMA and Exponential Smoothing were fit on the historical USD/AUD exchange rate series and evaluated on a 30-day out-of-sample holdout using MAE, RMSE, and MAPE. Missing calendar days (weekends/holidays with no quoted rate) were handled with **forward fill**, carrying the last known trading rate forward rather than interpolating, since no new rate information actually existed on those days.

The ARIMA model, built after confirming stationarity of the first-differenced series via the ADF test and selecting parameters guided by the ACF/PACF plots, captures short-term autocorrelation structure in the series. Its residual diagnostics (Ljung-Box test and residual ACF) were used to check that no significant autocorrelation remained unexplained.

Exponential Smoothing's trend configuration (`trend`, `damped_trend`) was selected via a **grid search over AIC** rather than fixed upfront, so the reported model reflects the best-fitting Holt's-method specification for this series rather than an arbitrary choice. It captures the overall trend in the series with fewer parameters and is computationally simpler than ARIMA, but does not explicitly model autocorrelation in the residuals the way ARIMA does.

Based on the error metrics table above, the model with the lower MAE, RMSE, and MAPE on the test set is the better performing model for this dataset. In general, for a highly persistent, near-random-walk series like a daily exchange rate, both models tend to perform similarly over short horizons, with ARIMA offering more flexibility if meaningful autocorrelation exists beyond the trend, and Exponential Smoothing offering a simpler, more robust baseline when the series behaves close to a random walk with drift.